# Weather API Exploration

## Objective

Learn how to retrieve data from a REST API and inspect the JSON response.

### Learning Goals

- API requests
- JSON responses
- API-based ingestion

In [0]:
import requests

In [0]:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast?latitude=40.7128&longitude=-74.0060&current=temperature_2m"
)

In [0]:
print(response.status_code)

## Parse API Response

Convert the API response into JSON format and inspect the returned structure.

In [0]:
weather_json = response.json()

In [0]:
type(weather_json)

In [0]:
weather_json.keys()

## Retrieve Daily Weather Data

Request multiple days of weather measurements that can later be joined with trip data.

In [0]:
weather_url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=40.7128"
    "&longitude=-74.0060"
    "&daily=temperature_2m_max,"
    "temperature_2m_min,"
    "precipitation_sum"
    "&forecast_days=14"
    "&timezone=America/New_York"
)

response = requests.get(weather_url)

print(response.status_code)

In [0]:
weather_json = response.json()

In [0]:
weather_json.keys()

In [0]:
weather_json["daily"]

## Convert Daily Weather Arrays Into Rows

Transform API arrays into a tabular structure that can be loaded into Spark.

In [0]:
daily_weather = weather_json["daily"]

In [0]:
daily_weather.keys()

In [0]:
weather_records = list(
    zip(
        daily_weather["time"],
        daily_weather["temperature_2m_max"],
        daily_weather["temperature_2m_min"],
        daily_weather["precipitation_sum"]
    )
)

In [0]:
weather_records[:5]

## Create Spark DataFrame

Convert weather records into a Spark DataFrame for further processing and storage.

In [0]:
weather_df = spark.createDataFrame(
    weather_records,
    [
        "weather_date",
        "max_temperature",
        "min_temperature",
        "precipitation"
    ]
)

In [0]:
display(weather_df)

In [0]:
weather_df.printSchema()

## Add Ingestion Metadata

Capture source and load information for traceability and auditing.

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

bronze_weather_df = (
    weather_df
    .withColumn(
        "source_system",
        lit("open-meteo-api")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

In [0]:
display(bronze_weather_df)

## Create Bronze Delta Table

Persist API weather data in the Bronze layer.

In [0]:
bronze_weather_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.bronze_weather"
    )

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_weather

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_weather